# The +1 / 0 / -1 signal per stock (train and test)

**What the professor asks** ([resources/instructions.pdf](../resources/instructions.pdf), "Project Overview & Core Task"):
1. the system must give, for each stock and at each time, a signal **+1** (long), **0** (out of the market) or **-1** (short);
2. the return of each period is the **return of the stock multiplied by the signal**;
3. the assumptions must be stated.

**What we do here.** The strategy works on pairs and the backtest ([backtest.py](backtest.py)) saves trades. This notebook turns the same trades into a signal for each of the 30 stocks, minute by minute, and recomputes the profit with the professor's formula. It changes no rule, no trade and no decision of the agent. The versions are those compared on the test: **A** (no agent), **B** and **C** of `finbert_episode`, on both the train and the test.

**The main numbers are still those of the trades** (Steps 6 and 7 and their test notebooks). The computation from the signal is the same strategy written in the professor's format, and it also serves to check that the two computations match. This notebook only reads files already saved: it does not redo the test.

## The assumptions

- **When it is decided and when it is executed.** The signal is decided with the prices up to the close of the 1-minute bar *t* and executed at the open of bar *t+1*: no future data.
- **From which signal.** The trade enters when the z-score of the pair's spread goes beyond 2 in absolute value (after the first 15 minutes and before the last 30), and exits when it goes back to 0, with the stop at 3.5, after 90 minutes or at the end of the day. In B the agent can block the entry; in C it can also enter in the opposite direction, following the news. Full rules in [signals.py](signals.py) and [docs/Test.md](../docs/Test.md).
- **From pair to stock.** In a "long A / short B" trade stock A has **+1** and stock B has **-1**; outside the trades both have **0**. If two trades of the same pair were open together, the positions would add up (the group's choice); with `finbert_episode` it never happens (section 3).
- **Return of a period.** The signal of bar *t* applies to the simple return of the stock from the open of bar *t* to the open of bar *t+1*; in the last bar of the day, from the open to the close. At night all signals are 0: no open position.
- **Amounts.** 10,000 dollars on each stock with a non-zero signal, no leverage; capital of 20,000 dollars per pair, 300,000 in total. The daily return is the dollar profit of the day divided by 300,000, without reinvestment (as in the backtest).
- **Short selling** always possible and without borrowing costs. **Cost 0**, the main scenario (the professor's indication).

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, next(str(p) for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()))   # project folder: makes src importable

import numpy as np
import pandas as pd

from src.config import RESULTS, RESULTS_TEST, TEST_END, TEST_START, TRAIN_END, VARIANTS
from src.backtest import CAPITAL_PER_PAIR, NOTIONAL_LEG, load_trades
from src.signals import list_pairs, load_pair

pd.options.display.float_format = "{:,.2f}".format
pd.options.display.width = 220
prices = {p: load_pair(p) for p in list_pairs().pair}          # 1-minute prices of the 15 pairs, train and test
CAPITAL = len(prices) * CAPITAL_PER_PAIR                        # 300,000 dollars
all_days = pd.DatetimeIndex(next(iter(prices.values())).ts.normalize().unique())
DAYS = {"train": all_days[all_days <= TRAIN_END], "test": all_days[(all_days >= TEST_START) & (all_days <= TEST_END)]}
TRADES = {   # the trades saved by the backtest, per period and version
    "train": {"A": RESULTS / "trades_A.csv", "B": VARIANTS / "finbert_episode/trades_B.csv", "C": VARIANTS / "finbert_episode/trades_C.csv"},
    "test": {v: RESULTS_TEST / f"trades_{v}.csv" for v in "ABC"},
}
trade = {(per, v): load_trades(path, allow_test=per == "test") for per, d in TRADES.items() for v, path in d.items()}
print({k: len(t) for k, t in trade.items()}, f"| capital {CAPITAL:,} $ | days: train {len(DAYS['train'])}, test {len(DAYS['test'])}")

{('train', 'A'): 973, ('train', 'B'): 968, ('train', 'C'): 969, ('test', 'A'): 500, ('test', 'B'): 493, ('test', 'C'): 495} | capital 300,000 $ | days: train 64, test 32


## 1. From the trades to the signal

For each pair we start from a vector of zeros, one position per bar. For each trade +1 (long the first stock of the pair) or -1 (short) is added from the entry bar to the bar before the exit one; if the trade exits at the end of the day, also the last bar, because the exit is at the close. The second stock of the pair has the opposite signal.

In [2]:
def signal(trades: pd.DataFrame, days: pd.DatetimeIndex) -> pd.DataFrame:
    """Signal per stock (columns) and per 1-minute bar (rows) on the given days, from the trades of a version."""
    columns = {}
    for p, d in prices.items():
        pos = np.zeros(len(d.ts), dtype=int)
        for t in trades[trades.pair == p].itertuples():
            e, k = d.ts.get_loc(t.t_entry), d.ts.get_loc(t.t_exit)
            end = k + 1 if t.exit_reason == "end_of_day" else k     # exit at the close of k, or at the open of k
            pos[e:end] += 1 if t.direction.startswith("long") else -1
        keep = d.ts.normalize().isin(days)
        columns[d.sym_A] = pd.Series(pos[keep], index=d.ts[keep])
        columns[d.sym_B] = pd.Series(-pos[keep], index=d.ts[keep])
    return pd.DataFrame(columns).fillna(0).astype("int8").sort_index().rename_axis("ts")

## 2. From the signal to the return

The simple return of each stock per bar, as in the assumptions: from the open of the bar to the open of the next one, and in the last bar of the day from the open to the close. The dollar profit of a bar is the sum, over the 30 stocks, of signal x return x 10,000 dollars.

In [3]:
def returns(days: pd.DatetimeIndex) -> pd.DataFrame:
    """Simple return of each stock (columns) per 1-minute bar (rows) on the given days."""
    columns = {}
    for d in prices.values():
        keep = d.ts.normalize().isin(days)
        for sym, open_, close in ((d.sym_A, d.oA, d.cA), (d.sym_B, d.oB, d.cB)):
            next_open = np.r_[open_[1:], np.nan]
            r = np.where(d.last_of_day, close / open_ - 1, next_open / open_ - 1)
            columns[sym] = pd.Series(r[keep], index=d.ts[keep])
    return pd.DataFrame(columns).sort_index().rename_axis("ts")


def daily_profit(s: pd.DataFrame, r: pd.DataFrame, days: pd.DatetimeIndex) -> pd.DataFrame:
    """Dollar profit and return on capital per day, with the professor's formula. Days without positions count as 0."""
    pnl = (s * r[s.columns]).fillna(0).sum(axis=1) * NOTIONAL_LEG     # a pair without prices on a day has signal 0
    pnl = pnl.groupby(pnl.index.normalize()).sum().reindex(days, fill_value=0.0)
    return pd.DataFrame({"pnl_usd": pnl, "ret": pnl / CAPITAL}).rename_axis("day")

## 3. The signal of A, B and C, on train and test

For each period and version: how many trades, in how many minutes x stocks the signal is non-zero, which values it takes, and in how many bars two trades of the same pair are open together (there the signal could be +2 or -2).

In [4]:
signals = {k: signal(t, DAYS[k[0]]) for k, t in trade.items()}


def overlapping(t: pd.DataFrame) -> int:
    """Trades that start before the previous trade of the same pair is closed."""
    t = t.sort_values(["pair", "t_entry"])
    return int((t.t_entry < t.groupby("pair").t_exit.shift()).sum())


pd.DataFrame([dict(period=per, version=v, trade=len(trade[(per, v)]), ticker_minutes_with_position=int((s != 0).sum().sum()),
                   values=" ".join(map(str, sorted(np.unique(s.to_numpy())))), overlapping_trades=overlapping(trade[(per, v)]))
              for (per, v), s in signals.items()]).set_index(["period", "version"])

trade  ticker_minutes_with_position  values  overlapping_trades
period version                                                                 
train  A          973                        122784  -1 0 1                   0
       B          968                        121884  -1 0 1                   0
       C          969                        122064  -1 0 1                   0
test   A          500                         63170  -1 0 1                   0
       B          493                         62186  -1 0 1                   0
       C          495                         62474  -1 0 1                   0

An example: the signal in the first minutes in which the test A opens positions (only the stocks with a non-zero signal). At 9:46 on 17 September the strategy buys CRM and sells WDAY, sells KO and buys PEP, sells BAC and buys C.

In [5]:
s = signals[("test", "A")]
first = s.index[(s != 0).any(axis=1)][0]
example = s.loc[first - pd.Timedelta(minutes=2): first + pd.Timedelta(minutes=5)]
example.loc[:, (example != 0).any()]

,MA,V,CRM,WDAY,KO,PEP,BAC,C
ts,,,,,,,,
2026-09-17 09:44:00,0,0,0,0,0,0,0,0
2026-09-17 09:45:00,0,0,0,0,0,0,0,0
2026-09-17 09:46:00,0,0,1,-1,-1,1,-1,1
2026-09-17 09:47:00,0,0,1,-1,-1,1,-1,1
2026-09-17 09:48:00,0,0,1,-1,-1,1,-1,1
2026-09-17 09:49:00,0,0,1,-1,-1,1,-1,1
2026-09-17 09:50:00,0,0,1,-1,-1,1,-1,1
2026-09-17 09:51:00,1,-1,1,-1,-1,1,-1,1


## 4. The return with the professor's formula, next to the trades

Total profit at cost 0 computed in two ways: summing the trades of the backtest (`pnl_trade_usd`, the main numbers of the project) and with signal x stock return (`pnl_signal_usd`). Then the difference of B and C from A with each of the two computations.

Why the two numbers are not identical: the backtest measures the profit of a trade with the log prices and keeps the amounts decided at entry fixed; the professor's formula uses simple returns and constant weights minute by minute.

In [6]:
daily, rows = {}, []
for per in DAYS:
    r = returns(DAYS[per])
    for v in "ABC":
        g = daily_profit(signals[(per, v)], r, DAYS[per])
        daily[(per, v)] = g
        rows.append(dict(period=per, version=v, pnl_trade_usd=trade[(per, v)].gross_bps.sum() / 1e4 * NOTIONAL_LEG, pnl_signal_usd=g.pnl_usd.sum()))
comparison = pd.DataFrame(rows).set_index(["period", "version"])
comparison["difference_usd"] = comparison.pnl_signal_usd - comparison.pnl_trade_usd
for col, name in (("pnl_trade_usd", "delta_from_A_trade_usd"), ("pnl_signal_usd", "delta_from_A_signal_usd")):
    comparison[name] = comparison[col] - comparison[col].xs("A", level="version").reindex(comparison.index, level="period")
comparison

pnl_trade_usd  pnl_signal_usd  difference_usd  delta_from_A_trade_usd  delta_from_A_signal_usd
period version                                                                                                
train  A             7,239.43        7,245.45            6.02                    0.00                     0.00
       B             6,954.50        6,957.10            2.59                 -284.92                  -288.35
       C             6,588.66        6,597.63            8.98                 -650.77                  -647.82
test   A             2,472.42        2,477.40            4.98                    0.00                     0.00
       B             2,283.56        2,289.28            5.73                 -188.86                  -188.12
       C             1,229.96        1,232.21            2.24               -1,242.46                -1,245.19

## 5. Saving

- the signal (minutes x 30 stocks) next to the trades of each version: train in `Results/signal_A.parquet` and `Results/agent_variants/finbert_episode/signal_B.parquet`, `signal_C.parquet`; test in `Results/test/signal_A.parquet`, `signal_B.parquet`, `signal_C.parquet`;
- the daily return from the signal of A, B and C: `Results/signal_returns.csv` (train) and `Results/test/signal_returns.csv` (test);
- the comparison table, train and test: `Results/signal_vs_trade_comparison.csv`.

In [7]:
WHERE = {("train", "A"): RESULTS, ("train", "B"): VARIANTS / "finbert_episode", ("train", "C"): VARIANTS / "finbert_episode"} | {("test", v): RESULTS_TEST for v in "ABC"}
for (per, v), s in signals.items():
    path = WHERE[(per, v)] / f"signal_{v}.parquet"
    s.to_parquet(path)
    print(path.relative_to(RESULTS.parent), s.shape)
for per, folder in (("train", RESULTS), ("test", RESULTS_TEST)):
    path = folder / "signal_returns.csv"
    pd.concat({v: daily[(per, v)].ret for v in "ABC"}, axis=1).to_csv(path)
    print(path.relative_to(RESULTS.parent))
comparison.to_csv(RESULTS / "signal_vs_trade_comparison.csv")
print((RESULTS / "signal_vs_trade_comparison.csv").relative_to(RESULTS.parent))

Results/signal_A.parquet (24960, 30)
Results/agent_variants/finbert_episode/signal_B.parquet (24960, 30)
Results/agent_variants/finbert_episode/signal_C.parquet (24960, 30)
Results/test/signal_A.parquet (12480, 30)
Results/test/signal_B.parquet (12480, 30)
Results/test/signal_C.parquet (12480, 30)
Results/signal_returns.csv
Results/test/signal_returns.csv
Results/signal_vs_trade_comparison.csv


## 6. What the results tell us

**The signal exists and has only three values.** For each of the 30 stocks and for every minute, on the train (64 days) and on the test (32 days), the signal is -1, 0 or +1. No trade of `finbert_episode` overlaps another of the same pair, neither in A nor in B nor in C: the sum rule never fires.

**The professor's formula gives the same results as the trades.** The difference goes from +2 to +9 dollars over the whole period, less than 0.3% of the profit of each version (for example the test A: 2,472 dollars from the trades, 2,477 from the signal).

**The conclusions do not change.** The differences of B and C from A are the same to within 3 dollars:

| | From the trades | From the signal |
|---|---|---|
| Test, B minus A | -189 $ | -188 $ |
| Test, C minus A | -1,242 $ | -1,245 $ |
| Train, B minus A | -285 $ | -288 $ |
| Train, C minus A | -651 $ | -648 $ |

What Steps 6 and 7 say about the strategy and the agent also holds with the signal and the professor's formula.

**What the saved files are for.** The daily returns from the signal (`signal_returns.csv`) are the base for the equity curve from 100 with the benchmark and for the annualised metrics.

**Limits.**
- The same assumptions as the backtest: short selling always possible and free, execution at the open of the next bar without price impact.
- Cost 0 only: with costs the computation from the signal should be repeated charging every change of signal.
- Intraday prices are not adjusted for dividends (Step 2): the same limit holds for the signal.